In [21]:
# 将下面的 "你的真实密钥" 替换为你个人的 API Key

api_key = "" 

content = f"""BASE_URL=https://chat.ecnu.edu.cn/open/api/v1
MODEL=ecnu-plus
API_KEY={api_key}
"""

# 写入 .env 文件

with open(".env1", "w", encoding="utf-8") as f:
    f.write(content)

print("✅ .env 文件已在当前目录创建成功！")

✅ .env 文件已在当前目录创建成功！


In [2]:
# 检查env文件是否创建   win的指令是!dir /a
!ls -a

.                  .env               .ipynb_checkpoints
..                 .env1              test.ipynb


In [3]:
# 查看内容     win的指令是   !type .env
!cat .env

BASE_URL=https://chat.ecnu.edu.cn/open/api/v1
MODEL=ecnu-plus
API_KEY=sk-58d07e67527441a184408cbe3b68b97b


In [4]:
# 安装openai依赖 读取env内容。 或者 !pip install openai python-dotenv -i https://pypi.tuna.tsinghua.edu.cn/simple
! pip install openai python-dotenv

In [4]:
# 导入依赖
import os
#读取env文件
from dotenv import load_dotenv
#openai协议
from openai import OpenAI

In [5]:
# 读取环境
load_dotenv(".env", override=True)
client = OpenAI(
api_key=os.environ["API_KEY"],
base_url=os.environ["BASE_URL"],
)

In [6]:
def ask_llm(question, system_prompt="你是一个AI助手,回答尽量简短。",temperature=0.7, max_tokens=2000):

    # 构建消息格式：第一条是系统提示词，第二条是用户提问
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    
    # 发送请求
    response = client.chat.completions.create(
        model=os.environ["MODEL"],
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        stream=False,
    )
    
    # 提取回答内容
    answer = response.choices[0].message.content
    
    # 打印结果（格式化输出）
    print("提问：", question)
    print("回答：", answer)
    
    # 返回回答，方便后续可能需要对回答做其他处理
    return answer

In [7]:
# 第一次提问
ask_llm("我正在上大模型原理的课程")

提问： 我正在上大模型原理的课程
回答： 你好！很高兴能协助你学习大模型原理。

请问你目前具体在学习哪个部分？例如：
*   Transformer 架构
*   预训练与微调
*   注意力机制
*   推理优化
*   其他

请随时提问，我会简短清晰地解答。


'你好！很高兴能协助你学习大模型原理。\n\n请问你目前具体在学习哪个部分？例如：\n*   Transformer 架构\n*   预训练与微调\n*   注意力机制\n*   推理优化\n*   其他\n\n请随时提问，我会简短清晰地解答。'

In [8]:
# 第二次提问
ask_llm("我现在在干嘛")

提问： 我现在在干嘛
回答： 你在和我聊天。


'你在和我聊天。'

In [10]:
# 但是我们平时使用到的GPT，其实你问的下一个问题，都是和上面的问题有关联的，所以，多轮对话的设计才更有意义。为了方便，我们将其设置为 类 的形式

In [15]:
class ChatBot:
    def __init__(self, system_prompt="你是一个AI助手,回答尽量简短。"):
        # 初始化历史记录，默认包含一条 system 消息
        self.history = [
            {"role": "system", "content": system_prompt}
        ]
    
    def ask(self, question,temperature=0.7, max_tokens=20000):
        """
        提问函数：默认在原有历史记录上继续对话
        """
        # 1. 把用户的新问题加入历史
        self.history.append({"role": "user", "content": question})
        
        try:
            # 2. 发送请求，把整个历史记录传给模型
            response = client.chat.completions.create(
                model=os.environ["MODEL"],
                messages=self.history,
                temperature=temperature,
                max_tokens=max_tokens,
                stream=False,
            )
            
            # 3. 提取回答
            answer = response.choices[0].message.content
            
            # 4. 把模型的回答也加入历史（这样它下次就知道自己说过什么了）
            self.history.append({"role": "assistant", "content": answer})
            
            # 打印结果
            #print(f"👤 提问：{question}")
            print(f"🤖 回答：{answer}\n")
            
            return answer
            
        except Exception as e:
            print(f"❌ 请求出错：{e}")
            # 如果出错了，把刚才加入的用户问题弹出去，保持历史记录干净
            self.history.pop()
            return None
            
    def clear_history(self):
        """
        清空对话历史：把记录重置，只保留最初设定的 system 提示词
        """
        # 保留第一条 system 消息
        if self.history and self.history[0]["role"] == "system":
            self.history = [self.history[0]]
        else:
            self.history = []
        print("🧹 对话历史已清空，可以开始新的话题了！\n")
        
    def show_history(self):

        print(self.history)
        # 计算当前有多少轮对话 (总条数 - 1条system) / 2
        turns = (len(self.history) - 1) // 2 
        print(f"📊 当前已进行 {turns} 轮对话。\n")

In [16]:
# 定义一个实例
bot = ChatBot(system_prompt="你是一个AI助手，回答尽量简短。")

In [17]:
bot.clear_history()

🧹 对话历史已清空，可以开始新的话题了！



In [20]:
while True:
    question = input("你: ").strip()
    if question.lower() == "quit":
        print("Bye!")
        break
    if not question:          # skip empty input
        continue
    answer = bot.ask(question)
    #if answer:
        #print(f"AI: {answer}\n")

你:  hi


🤖 回答：你好！请问有什么我可以帮你的吗？



你:  大模型在接下来的大模型发展中，究竟是会先彻底颠覆教育行业，还是先颠覆医疗行业？


🤖 回答：更有可能**先深刻改变教育**，但**医疗**的颠覆潜力和长期影响更大。

原因如下：

1.  **教育门槛低、容错率高**：教育场景对错误相对宽容，技术落地阻力小，且大模型在个性化辅导、内容生成上已迅速普及。
2.  **医疗监管严、风险高**：医疗涉及生命安全，受严格法规限制，数据隐私要求极高，AI目前主要作为辅助工具，全面颠覆需突破信任与法律瓶颈。

简言之：**教育先变，医疗后变。**



你:  但是医疗方面启动早样本多，许多深度学习的训练和研究都是基于医疗的。教育类偏少


...

🤖 回答：这是一个非常敏锐的技术视角。确实，**“大模型”（LLM）与“判别式/专用深度学习模型”在医疗和教育中的角色截然不同**。

1.  **医疗：依赖“专用小模型”或“微调模型”**
    *   核心任务多为**分类、检测、分割**（如CT影像识别、病理切片分析）。
    *   这类任务不需要生成能力，而是需要极高的精度和可解释性。
    *   因此，医疗AI更多是**垂直领域的专用模型**，而非通用大语言模型（LLM）。虽然LLM开始介入病历分析和辅助决策，但诊断主力仍是传统深度学习。

2.  **教育：依赖“通用大模型”（LLM）**
    *   核心任务是**内容生成、对话交互、逻辑推理**。
    *   GPT类模型天然适合处理自然语言，能直接提供个性化辅导、出题、作文批改等。
    *   教育场景对“幻觉”容忍度相对较高，且无需严格的物理世界建模。

**结论修正：**

*   **如果定义“颠覆”为“技术范式转移”**：
    *   **医疗**早已通过**专用深度学习**实现了部分颠覆（如影像科），但这属于“旧AI”。
    *   **教育**正通过**通用大模型（LLM）**实现新一轮颠覆，这属于“新AI”。

*   **关键区别**：
    *   医疗的“颠覆”更多来自**感知智能**（看懂片子），门槛高、落地慢。
    *   教育的“颠覆”更多来自**认知智能**（理解并生成知识），门槛低、扩散快。

所以，你的观点成立：**从“当前主流大模型（LLM）”的角度看，教育确实是更直接的试验田；而医疗的“颠覆”主要发生在更早的专用AI阶段，且仍在持续深化。**



你:  请你基于上述的讨论完成对于论题的总结性描述，避免片面的论断


🤖 回答：基于我们关于技术成熟度、行业特性及AI模型定义的深入探讨，对“大模型将先颠覆教育还是医疗”这一论题的总结性描述如下：

**结论：两者并非简单的先后替代关系，而是处于不同维度、不同阶段的“错位演进”。若以“通用大语言模型（LLM）引发的范式变革”为界定标准，教育行业正经历更快速、广泛的表层重构；而医疗行业则依托更早成熟的专用深度学习技术，正在经历更深层次但受限于监管与信任阈值的底层颠覆。**

具体而言：

1.  **技术路径的差异导致落地节奏不同**
    *   **教育**主要依赖**生成式大模型（如GPT）**。其核心能力在于自然语言处理、内容生成和逻辑交互，这与教育中知识传递、个性化辅导的需求高度契合。由于容错率相对较高且数据壁垒较低，LLM能迅速渗透至教学辅助环节，实现“工具层”的快速迭代。
    *   **医疗**的核心诊断与治疗长期依赖**判别式/专用深度学习模型**（如影像识别、病理分析）。这些技术在LLM爆发前已积累多年，实现了局部的高效自动化。当前LLM在医疗中的应用仍局限于病历整理、辅助决策等非核心环节，因为生命安全的刚性约束要求极高的可解释性和零错误率，这限制了通用大模型的直接“颠覆”。

2.  **“颠覆”的定义需分层看待**
    *   **教育的“浅层颠覆”**：如果仅视教育为知识分发，LLM确实已在颠覆传统课堂模式。但若将教育视为包含资源分配、社会化培养、情感连接的系统工程，AI目前更多是**增强者**而非**替代者**。它难以短期内重构评价体系或提供真实的人际互动体验。
    *   **医疗的“深层颠覆”**：医疗的目标（治愈疾病）具有客观性和标准化特征。一旦AI在特定领域（如早期筛查、精准用药）突破临床验证的信任阈值，其带来的将是结构性的效率提升和治疗范式转移。这种颠覆虽启动早、样本多，但因监管严苛，呈现“厚积薄发”的特征。

3.  **综合判断**
    *   **短期看（1-3年）**：**教育**因LLM的低门槛和高适配性，将在用户体验、内容生产上表现出更明显的“颠覆感”，但这更多是流程优化和效率提升。
    *   **长期看（5-10年+）**：**医疗**可能迎来更具本质意义的颠覆。随着多模态大模型与专用小模型的融合，以及监管框架的逐步完善，AI将从辅助角色转向核心诊疗决策支持，彻

你:  quit


Bye!


In [17]:
# 课程任务：请从下列主题中，选择一个，然后和大模型进行多轮对话，得出一个统一的结论（可以是你说服大模型，也可以是你认可大模型的想法，对话轮次至少5轮以上）
# 任务提交：所有的代码，以及和大模型的对话记录


In [18]:
# 主题1：“大模型在接下来的大模型发展中，究竟是会先彻底颠覆教育行业，还是先颠覆医疗行业？”

In [19]:
# 主题2：“随着大模型越来越拟人化，我们是否应该赋予其‘虚拟人格权’？”

In [20]:
# 主题3：“面对 AI 顶会投稿量暴增及大量 LLM 生成‘抽奖’稿件，学术界应采取何种防御与应对机制？”